# QC 3D · Huấn luyện model phát hiện 3D theo class giao thông VN (Google Colab)

Model: **CenterPoint (pillar)** trên OpenPCDet, học tiếp từ model đã huấn luyện trên toàn bộ nuScenes;
class: Car, Truck, Bus, Rider, Motorbike (xe 2 bánh đỗ), Pedestrian.
Dữ liệu: file zip do `utils/qc3d/run_after_download.py` tạo (PHENIKAA + nuScenes-mini).

Cách chạy:
1. Upload `train_vn_kaggle_v4.zip` (hoặc bản mới nhất) lên Google Drive, vào thư mục **`qc3d`** (tạo thư mục này ở *My Drive*).
2. Menu *Runtime* → *Change runtime type* → **T4 GPU** → *Save*.
3. *Runtime* → **Run all**; khi được hỏi, cho phép notebook truy cập Google Drive.
4. Giữ tab trình duyệt mở. Mỗi 5 phút notebook in một dòng tiến độ.
5. Nếu Colab ngắt giữa chừng: kết nối lại → *Run all*. Checkpoint nằm trên Drive nên học tiếp từ epoch đã lưu
   (mất khoảng 15 phút cài lại OpenPCDet). Muốn học lại từ đầu: đặt `START_FRESH = True`.
6. Kết quả (checkpoint, log, điểm đánh giá) ở Google Drive: `qc3d/results/`.

In [ ]:
EPOCHS = 30         # 2 để chạy thử nhanh; 30 cho bản v2 (v1: loss vẫn giảm ở epoch 20)
BATCH_SIZE = 4      # giảm còn 2 nếu báo hết bộ nhớ GPU (CUDA out of memory)
CLASSES = ['Car', 'LargeVehicle', 'TwoWheeler', 'Pedestrian']
DRIVE_DIR = '/content/drive/MyDrive/qc3d'   # chứa file zip dữ liệu; checkpoint và kết quả cũng lưu ở đây
START_FRESH = False  # True: xoá checkpoint cũ trên Drive và học lại từ đầu

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys, torch
print(sys.version)
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'gpu', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('Chưa bật GPU: menu Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save')

## 1. Cài OpenPCDet (biên dịch CUDA, khoảng 10-15 phút)

In [ ]:
%cd /content
!test -d OpenPCDet || git clone --depth 1 https://github.com/open-mmlab/OpenPCDet.git
!pip install -q easydict tensorboardX pyquaternion numba scikit-image ninja
!pip install -q SharedArray || echo "SharedArray skipped (only needed by other datasets)"

In [ ]:
# spconv: needed only by voxel (sparse conv) models, but OpenPCDet imports it at start-up.
# Pillar models never call it, and the real wheels JIT-compile `cumm` on import, which fails on
# recent Python versions (missing tensorview headers): always use a stub instead.
import site, subprocess, sys
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'spconv', 'cumm',
                'spconv-cu126', 'spconv-cu124', 'spconv-cu121', 'spconv-cu120',
                'cumm-cu126', 'cumm-cu124', 'cumm-cu121', 'cumm-cu120'], capture_output=True)
STUB = (
    '"""Stub of spconv: imported by OpenPCDet, never used by pillar models."""\n'
    'import types as _types\n'
    'import torch.nn as _nn\n'
    "__version__ = '2.3.6'\n\n\n"
    'def _resolve(owner, name):\n'
    "    # lower case: a sub-module (spconv.conv, spconv.pytorch.functional); upper case: a layer class\n"
    "    if name.startswith('__'):\n"
    '        raise AttributeError(name)\n'
    '    value = _Namespace(name) if name[:1].islower() else type(name, (_nn.Module,), {})\n'
    '    setattr(owner, name, value)\n'
    '    return value\n\n\n'
    'class _Namespace(_types.ModuleType):\n'
    '    def __getattr__(self, name):\n'
    '        return _resolve(self, name)\n\n\n'
    'class SparseModule(_nn.Module):\n'
    '    pass\n\n\n'
    'def __getattr__(name):\n'
    '    import sys\n'
    '    return _resolve(sys.modules[__name__], name)\n'
)
import importlib.util, shutil
spec = importlib.util.find_spec('spconv')  # a leftover real package must be replaced where it is found
if spec and spec.origin:
    shutil.rmtree(Path(spec.origin).parent, ignore_errors=True)
root = Path(spec.origin).parent if spec and spec.origin else Path(site.getsitepackages()[0]) / 'spconv'
(root / 'pytorch').mkdir(parents=True, exist_ok=True)
(root / '__init__.py').write_text(STUB)
(root / 'pytorch' / '__init__.py').write_text(STUB)
# spconv.utils: the point -> pillar generator used by OpenPCDet's data processor (numpy, utils/qc3d/kaggle)
(root / 'utils').mkdir(exist_ok=True)
(root / 'utils' / '__init__.py').write_text('# Copyright (C) CVAT.ai Corporation\n#\n# SPDX-License-Identifier: MIT\n\n"""\nNumpy point -> voxel (pillar) generator with the spconv 1.x API that OpenPCDet\'s data processor uses:\n    VoxelGeneratorV2(voxel_size, point_cloud_range, max_num_points, max_voxels).generate(points)\n    -> (voxels [M, max_num_points, F], coordinates [M, 3] as (z, y, x), num_points_per_voxel [M])\n\nInstalled as `spconv.utils` in the training notebooks (utils/qc3d/kaggle/make_notebook.py), where the real\nspconv cannot be imported, and used by the QC detector (detector.py) to run the model without a GPU.\nVoxels keep the order in which they first appear in the (shuffled) point list, like spconv; points\nbeyond max_num_points in a voxel and voxels beyond max_voxels are dropped, like spconv.\n"""\n\nimport numpy as np\n\n\nclass VoxelGeneratorV2:\n    def __init__(self, voxel_size, point_cloud_range, max_num_points, max_voxels=20000, **_):\n        self.voxel_size = np.asarray(voxel_size, dtype=np.float32)\n        self.point_cloud_range = np.asarray(point_cloud_range, dtype=np.float32)\n        self.grid_size = np.round(\n            (self.point_cloud_range[3:] - self.point_cloud_range[:3]) / self.voxel_size\n        ).astype(np.int64)\n        self.max_num_points = int(max_num_points)\n        self.max_voxels = int(max_voxels)\n\n    def generate(self, points, max_voxels=None):\n        max_voxels = int(max_voxels or self.max_voxels)\n        index = np.floor((points[:, :3] - self.point_cloud_range[:3]) / self.voxel_size).astype(np.int64)\n        inside = np.all((index >= 0) & (index < self.grid_size), axis=1)\n        points, index = points[inside], index[inside]\n\n        nx, ny, _ = self.grid_size\n        key = (index[:, 2] * ny + index[:, 1]) * nx + index[:, 0]\n        _, first, inverse = np.unique(key, return_index=True, return_inverse=True)\n        inverse = inverse.reshape(-1)\n        kept = np.argsort(first, kind="stable")[:max_voxels]  # voxels in order of first appearance\n        rank = np.full(len(first), -1, dtype=np.int64)\n        rank[kept] = np.arange(len(kept))\n        voxel_id = rank[inverse]\n        keep = voxel_id >= 0\n        points, index, voxel_id = points[keep], index[keep], voxel_id[keep]\n\n        order = np.argsort(voxel_id, kind="stable")  # keeps the point order inside each voxel\n        voxel_id, points, index = voxel_id[order], points[order], index[order]\n        count = len(kept)\n        starts = np.searchsorted(voxel_id, np.arange(count))\n        slot = np.arange(len(voxel_id)) - starts[voxel_id]\n        fits = slot < self.max_num_points\n\n        voxels = np.zeros((count, self.max_num_points, points.shape[1]), dtype=points.dtype)\n        voxels[voxel_id[fits], slot[fits]] = points[fits]\n        num_points = np.minimum(np.bincount(voxel_id, minlength=count), self.max_num_points).astype(np.int32)\n        coordinates = np.zeros((count, 3), dtype=np.int32)\n        coordinates[voxel_id] = index[:, ::-1]  # (z, y, x)\n        return voxels, coordinates, num_points\n\n\nVoxelGenerator = VoxelGeneratorV2\n')
check = subprocess.run([sys.executable, '-c', (
    'import numpy as np, spconv, spconv.pytorch; from spconv.utils import VoxelGeneratorV2 as G; '
    'g = G([0.2, 0.2, 8.0], [-51.2, -51.2, -5.0, 51.2, 51.2, 3.0], 20, 40000); '
    'v, c, n = g.generate(np.array([[1.0, 2.0, 0.0, 9.0, 0.0]], np.float32)); '
    'print(spconv.__doc__, v.shape, c.tolist(), n.tolist())')], capture_output=True, text=True)
assert 'Stub of spconv' in check.stdout and '[[0, 266, 261]]' in check.stdout, check.stdout + check.stderr
print('spconv: stub + numpy voxel generator at', root)

In [ ]:
# OpenPCDet imports every dataset at start-up; datasets we do not use may need packages that are not
# installed (waymo, argo2, ...): make those imports optional.
import re
from pathlib import Path
init = Path('/content/OpenPCDet/pcdet/datasets/__init__.py')
text = init.read_text()
if 'QC3D_OPTIONAL' not in text:
    def optional(match):
        line = match.group(0)
        if 'custom' in line or 'dataset import DatasetTemplate' in line:
            return line
        name = re.search(r'import (\w+)', line).group(1)
        return f'try:\n    {line}\nexcept Exception:  # QC3D_OPTIONAL\n    {name} = None'
    text = re.sub(r'^from \.[\w.]+ import \w+$', optional, text, flags=re.M)
    text = re.sub(r"^(\s+)'(\w+)': (\w+),$", lambda m: m.group(0) if m.group(3) != 'None' else '', text, flags=re.M)
    init.write_text(text)

# numpy 2 removed np.int / np.float / np.bool / np.long, still used in places by OpenPCDet
changed = 0
for path in Path('/content/OpenPCDet/pcdet').rglob('*.py'):
    source = path.read_text()
    fixed = re.sub(r'\bnp\.long\b(?!\w)', 'np.int64', source)
    fixed = re.sub(r'\bnp\.(int|float|bool)\b(?!\w)', r'\1', fixed)
    if fixed != source:
        path.write_text(fixed)
        changed += 1
print('numpy aliases fixed in', changed, 'files')

# custom_dataset.py imports Path only under `if __name__ == '__main__'`, but uses it in functions we call
custom = Path('/content/OpenPCDet/pcdet/datasets/custom/custom_dataset.py')
source = custom.read_text()
if not re.search(r'^from pathlib import Path', source, flags=re.M):
    source = 'from pathlib import Path\n' + source
    print('custom_dataset.py: added `from pathlib import Path`')
# v2: label names -> the classes the model learns (vn_dataset.yaml CLASS_GROUPS), when the labels are read
if 'QC3D_GROUPS' not in source:
    source = source.replace(
        'gt_names.append(line_list[-1])',
        "gt_names.append(QC3D_GROUPS.get(line_list[-1], line_list[-1]))",
    )
    assert 'QC3D_GROUPS.get' in source, 'custom_dataset.get_label changed: cannot map the class groups'
    source = ("QC3D_GROUPS = {'Truck': 'LargeVehicle', 'Bus': 'LargeVehicle', 'Rider': 'TwoWheeler', "
              "'Motorbike': 'TwoWheeler'}\n" + source)
    print('custom_dataset.py: class groups', 'Truck/Bus -> LargeVehicle, Rider/Motorbike -> TwoWheeler')
# multi-sweep data (5th column = time lag): in training, half of the samples keep only the newest sweeps
# (sometimes one), so the model also works on a single sweep (CVAT jobs usually come without ego poses)
if 'QC3D_SWEEP_DROP' not in source:
    match = re.search(r'^(\s*)point_features = np\.load\(lidar_file\)$', source, flags=re.M)
    assert match, 'custom_dataset.get_lidar changed: cannot add the sweep dropout'
    pad = match.group(1)
    source = source.replace(match.group(0), match.group(0) + (
        f'\n{pad}if getattr(self, "training", False) and point_features.shape[1] > 4 '
        f'and np.random.rand() < QC3D_SWEEP_DROP:'
        f'\n{pad}    lags = np.unique(point_features[:, 4])'
        f'\n{pad}    point_features = point_features[point_features[:, 4] <= np.random.choice(lags)]'
    ))
    source = 'QC3D_SWEEP_DROP = 0.5\n' + source
    print('custom_dataset.py: sweep dropout 0.5 in training')
custom.write_text(source)

# v5, terrain steps (vn_dataset.yaml: random_region_z_shift): one side of a random line moves up or down with
# the objects on it, so the model also finds objects on a road lower or higher than the ego road
# (PHENIKAA val: 24 % of the cars within 30 m stand > 0.8 m below it, 2 % in the training data)
augmentor = Path('/content/OpenPCDet/pcdet/datasets/augmentor/data_augmentor.py')
source = augmentor.read_text()
if 'QC3D_REGION_Z' not in source:
    source += '''

def _qc3d_region_z_shift(self, data_dict=None, config=None):  # QC3D_REGION_Z
    if data_dict is None:
        return partial(_qc3d_region_z_shift, self, config=config)
    if np.random.rand() >= config['PROBABILITY']:
        return data_dict
    points, boxes = data_dict['points'], data_dict['gt_boxes']
    angle = np.random.uniform(0, 2 * np.pi)
    normal = np.array([np.cos(angle), np.sin(angle)], dtype=np.float32)
    line = np.random.uniform(*config['LINE_RANGE'])
    shift = np.random.uniform(*config['SHIFT_RANGE'])
    moved = points[:, :2] @ normal > line
    box_moved = boxes[:, :2] @ normal > line
    # a box across the line moves as a whole, with all its points
    across = np.abs(boxes[:, :2] @ normal - line) < np.hypot(boxes[:, 3], boxes[:, 4]) / 2 + 0.1
    for i in np.flatnonzero(across):
        x, y, z, dx, dy, dz, yaw = boxes[i, :7]
        rx, ry = points[:, 0] - x, points[:, 1] - y
        lx, ly = rx * np.cos(yaw) + ry * np.sin(yaw), -rx * np.sin(yaw) + ry * np.cos(yaw)
        inside = (np.abs(lx) <= dx / 2 + 0.1) & (np.abs(ly) <= dy / 2 + 0.1) & (np.abs(points[:, 2] - z) <= dz / 2 + 0.1)
        moved[inside] = box_moved[i]
    points[moved, 2] += shift
    boxes[box_moved, 2] += shift
    data_dict['points'], data_dict['gt_boxes'] = points, boxes
    return data_dict


DataAugmentor.random_region_z_shift = _qc3d_region_z_shift
'''
    augmentor.write_text(source)
    print('data_augmentor.py: random_region_z_shift (terrain steps)')

In [ ]:
# compile the CUDA ops in place (full log: /content/build.log) and make pcdet importable
%cd /content/OpenPCDet
import os, re, site, subprocess, sys
from pathlib import Path

# C++ APIs removed in recent PyTorch, still used by some OpenPCDet ops
for path in Path('pcdet/ops').rglob('*'):
    if path.suffix in ('.cpp', '.cu', '.h', '.cuh'):
        source = path.read_text()
        fixed = re.sub(r'\.type\(\)\.is_cuda\(\)', '.is_cuda()', source)
        fixed = re.sub(r'\.data<(\w+)>\(\)', r'.data_ptr<\1>()', fixed)
        if fixed != source:
            path.write_text(fixed)

import glob, shutil, torch
cuda_home = os.environ.get('CUDA_HOME')
if not cuda_home:  # the CUDA toolkit (nvcc) is needed to compile the ops
    nvcc = shutil.which('nvcc') or next(iter(sorted(glob.glob('/usr/local/cuda*/bin/nvcc'))), None)
    cuda_home = str(Path(nvcc).resolve().parents[1]) if nvcc else None
if not cuda_home:
    raise SystemExit('Không thấy CUDA toolkit (nvcc): phiên này chưa bật GPU. menu Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save, rồi chạy lại từ đầu')
arch = '%d.%d' % torch.cuda.get_device_capability()  # compile only for this GPU (T4 7.5, P100 6.0, L4 8.9, ...)
print('CUDA_HOME =', cuda_home, '| GPU arch', arch)
env = {**os.environ, 'CUDA_HOME': cuda_home, 'PATH': f"{cuda_home}/bin:" + os.environ.get('PATH', ''),
       'TORCH_CUDA_ARCH_LIST': arch, 'MAX_JOBS': '4'}
with open('/content/build.log', 'w') as log:
    proc = subprocess.run([sys.executable, 'setup.py', 'build_ext', '--inplace'],
                          stdout=log, stderr=subprocess.STDOUT, env=env)
built = sorted(p.name.split('.')[0] for p in Path('pcdet/ops').rglob('*.so'))
print(len(built), 'CUDA ops:', built)
if proc.returncode != 0 or not built:
    lines = open('/content/build.log').read().splitlines()
    print('\n'.join([l for l in lines if 'error' in l.lower()][:40] or lines[-40:]))
    raise SystemExit('Biên dịch OpenPCDet lỗi: gửi phần log ở trên')

(Path(site.getsitepackages()[0]) / 'openpcdet.pth').write_text('/content/OpenPCDet\n')
sys.path.insert(0, '/content/OpenPCDet')
import pcdet
from pcdet.ops.roiaware_pool3d import roiaware_pool3d_utils  # fails here if the ops are broken
print('pcdet', pcdet.__version__, 'OK')

## 2. Nối dữ liệu huấn luyện

In [ ]:
# Google Drive holds the zip (uploaded once) and the outputs, which survive a disconnected session
import os, shutil, zipfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

def find_root(base):
    for dirpath, dirnames, _ in os.walk(base, followlinks=True):
        if {'ImageSets', 'points', 'labels'} <= set(dirnames):
            return Path(dirpath)
    return None

DRIVE = Path(DRIVE_DIR)
DRIVE.mkdir(parents=True, exist_ok=True)
RESULTS = DRIVE / 'results'
src = find_root('/content/train_vn') if Path('/content/train_vn').exists() else None
if src is None:
    zips = sorted(DRIVE.glob('train_vn*.zip')) or sorted(Path('/content/drive/MyDrive').glob('train_vn*.zip'))
    if not zips:
        raise SystemExit('Không thấy file zip dữ liệu: upload train_vn_kaggle_v4.zip vào Google Drive, '
                         'thư mục My Drive/qc3d, rồi chạy lại cell này')
    archive = max(zips, key=lambda p: p.stat().st_mtime)
    # copy to the local disk first: extracting thousands of files straight from Drive is very slow
    print(f'copy {archive} ({archive.stat().st_size / 2**30:.2f} GB) → /content ...')
    local = Path('/content') / archive.name
    shutil.copy(archive, local)
    zipfile.ZipFile(local).extractall('/content/train_vn')
    local.unlink()
    src = find_root('/content/train_vn')
if src is None:
    raise SystemExit('File zip không có các thư mục points/, labels/, ImageSets/: dùng file do run_after_download.py tạo')
# labels_overlay.zip (labels/ + ImageSets/ only, a few MB): newer labels over the big data zip, no new upload of it
overlay = DRIVE / 'labels_overlay.zip'
if overlay.exists():
    zipfile.ZipFile(overlay).extractall(src)
    print('labels_overlay.zip:', len(zipfile.ZipFile(overlay).namelist()), 'file nhãn mới')
label_root = src

dst = Path('/content/OpenPCDet/data/custom')
dst.mkdir(parents=True, exist_ok=True)
for name in ('points', 'labels', 'ImageSets'):
    link = dst / name
    if not link.exists():
        link.symlink_to((src if name == 'points' else label_root) / name)
print('data:', src, '| labels:', label_root)
for split in ('train', 'val'):
    print(split, len((dst / 'ImageSets' / f'{split}.txt').read_text().split()), 'frames')

In [ ]:
%%writefile /content/OpenPCDet/tools/cfgs/dataset_configs/vn_dataset.yaml
# OpenPCDet dataset config for the VN training set built by utils/qc3d/build_train_set.py
DATASET: 'CustomDataset'
DATA_PATH: '../data/custom'

# ground at z = -1.8 (build_train_set.GROUND_Z); 102.4 m / 0.2 m = 512 cells (divisible by 8)
POINT_CLOUD_RANGE: [-51.2, -51.2, -5.0, 51.2, 51.2, 3.0]

# OpenPCDet evaluates custom datasets with the KITTI metric: classes are mapped to KITTI names
MAP_CLASS_TO_KITTI: {
    'Car': 'Car',
    'LargeVehicle': 'Truck',
    'TwoWheeler': 'Cyclist',
    'Pedestrian': 'Pedestrian',
}

# label names of the training set -> classes the model learns (applied by the notebook when reading labels)
CLASS_GROUPS: {
    'Truck': 'LargeVehicle',
    'Bus': 'LargeVehicle',
    'Rider': 'TwoWheeler',
    'Motorbike': 'TwoWheeler',
}

DATA_SPLIT: {
    'train': train,
    'test': val
}

INFO_PATH: {
    'train': [custom_infos_train.pkl],
    'test': [custom_infos_val.pkl],
}

# same 5 features as nuScenes (intensity 0-255, time lag 0) so nuScenes-pretrained weights fit as is
POINT_FEATURE_ENCODING: {
    encoding_type: absolute_coordinates_encoding,
    used_feature_list: ['x', 'y', 'z', 'intensity', 'timestamp'],
    src_feature_list: ['x', 'y', 'z', 'intensity', 'timestamp'],
}

DATA_AUGMENTOR:
    DISABLE_AUG_LIST: ['placeholder']
    AUG_CONFIG_LIST:
        - NAME: gt_sampling
          USE_ROAD_PLANE: False
          DB_INFO_PATH:
              - custom_dbinfos_train.pkl
          PREPARE: {
             filter_by_min_points: ['Car:5', 'LargeVehicle:5', 'TwoWheeler:5', 'Pedestrian:5'],
          }
          # the rarer large vehicles are pasted more often into training scenes
          SAMPLE_GROUPS: ['Car:10', 'LargeVehicle:10', 'TwoWheeler:10', 'Pedestrian:12']
          NUM_POINT_FEATURES: 5
          DATABASE_WITH_FAKELIDAR: False
          REMOVE_EXTRA_WIDTH: [0.0, 0.0, 0.0]
          LIMIT_WHOLE_SCENE: True

        # terrain steps: one side of a random line moves up or down with its objects (added by the notebook,
        # make_notebook.py); objects on a lower or higher road than the ego road
        - NAME: random_region_z_shift
          PROBABILITY: 0.5
          LINE_RANGE: [-25.0, 25.0]
          SHIFT_RANGE: [-3.0, 1.5]

        - NAME: random_world_translation
          NOISE_TRANSLATE_STD: [0.0, 0.0, 0.3]

        - NAME: random_world_flip
          ALONG_AXIS_LIST: ['x', 'y']

        - NAME: random_world_rotation
          WORLD_ROT_ANGLE: [-0.78539816, 0.78539816]

        - NAME: random_world_scaling
          WORLD_SCALE_RANGE: [0.95, 1.05]

DATA_PROCESSOR:
    - NAME: mask_points_and_boxes_outside_range
      REMOVE_OUTSIDE_BOXES: True

    - NAME: shuffle_points
      SHUFFLE_ENABLED: {
        'train': True,
        'test': False
      }

    - NAME: transform_points_to_voxels
      VOXEL_SIZE: [0.2, 0.2, 8.0]
      MAX_POINTS_PER_VOXEL: 20
      MAX_NUMBER_OF_VOXELS: {
        'train': 40000,
        'test': 40000
      }

In [ ]:
!mkdir -p /content/OpenPCDet/tools/cfgs/custom_models

In [ ]:
%%writefile /content/OpenPCDet/tools/cfgs/custom_models/vn_centerpoint_pillar.yaml
# CenterPoint (pillar) for the VN classes. Fits a 16 GB GPU (Kaggle T4 / P100) at batch size 4.
# Same layers as OpenPCDet's nuScenes cbgs_dyn_pp_centerpoint, so it is fine-tuned from those weights
# (static PillarVFE instead of DynPillarVFE: same parameters, no torch_scatter needed).
#
# v2: the model learns 4 groups the data supports well; QC splits them with the guideline rules, as R11:
#   LargeVehicle -> Truck / Bus by size (PHENIKAA has no Truck / Bus labels: they come from the box size,
#                   so the model cannot learn a better split; v1 found 100 % of near trucks but called them Bus)
#   TwoWheeler   -> Rider / Motorbike by height (only 108 parked two-wheelers to train a Motorbike class)
# The notebook maps the label names of the training set (vn_dataset.yaml CLASS_GROUPS): no new upload.
CLASS_NAMES: ['Car', 'LargeVehicle', 'TwoWheeler', 'Pedestrian']

DATA_CONFIG:
    _BASE_CONFIG_: cfgs/dataset_configs/vn_dataset.yaml

MODEL:
    NAME: CenterPoint

    VFE:
        NAME: PillarVFE
        WITH_DISTANCE: False
        USE_ABSLOTE_XYZ: True
        USE_NORM: True
        NUM_FILTERS: [64, 64]

    MAP_TO_BEV:
        NAME: PointPillarScatter
        NUM_BEV_FEATURES: 64

    BACKBONE_2D:
        NAME: BaseBEVBackbone
        LAYER_NUMS: [3, 5, 5]
        LAYER_STRIDES: [2, 2, 2]
        NUM_FILTERS: [64, 128, 256]
        UPSAMPLE_STRIDES: [0.5, 1, 2]
        NUM_UPSAMPLE_FILTERS: [128, 128, 128]

    DENSE_HEAD:
        NAME: CenterHead
        CLASS_AGNOSTIC: False
        # classes of similar size share a head
        CLASS_NAMES_EACH_HEAD: [
            ['Car'],
            ['LargeVehicle'],
            ['TwoWheeler'],
            ['Pedestrian'],
        ]
        SHARED_CONV_CHANNEL: 64
        USE_BIAS_BEFORE_NORM: True
        NUM_HM_CONV: 2
        SEPARATE_HEAD_CFG:
            HEAD_ORDER: ['center', 'center_z', 'dim', 'rot']
            HEAD_DICT: {
                'center': {'out_channels': 2, 'num_conv': 2},
                'center_z': {'out_channels': 1, 'num_conv': 2},
                'dim': {'out_channels': 3, 'num_conv': 2},
                'rot': {'out_channels': 2, 'num_conv': 2},
            }

        TARGET_ASSIGNER_CONFIG:
            FEATURE_MAP_STRIDE: 4
            NUM_MAX_OBJS: 500
            GAUSSIAN_OVERLAP: 0.1
            MIN_RADIUS: 2

        LOSS_CONFIG:
            LOSS_WEIGHTS: {
                'cls_weight': 1.0,
                'loc_weight': 0.25,
                'code_weights': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
            }

        POST_PROCESSING:
            SCORE_THRESH: 0.1
            POST_CENTER_LIMIT_RANGE: [-61.2, -61.2, -10.0, 61.2, 61.2, 10.0]
            MAX_OBJ_PER_SAMPLE: 500
            NMS_CONFIG:
                NMS_TYPE: nms_gpu
                NMS_THRESH: 0.2
                NMS_PRE_MAXSIZE: 1000
                NMS_POST_MAXSIZE: 200  # per head; dense VN traffic: 83 cut the two-wheelers

    POST_PROCESSING:
        RECALL_THRESH_LIST: [0.3, 0.5, 0.7]
        EVAL_METRIC: kitti

OPTIMIZATION:
    BATCH_SIZE_PER_GPU: 4
    NUM_EPOCHS: 30  # v1 loss still went down at epoch 20

    OPTIMIZER: adam_onecycle
    LR: 0.001  # fine-tuning from nuScenes weights
    WEIGHT_DECAY: 0.01
    MOMENTUM: 0.9

    MOMS: [0.95, 0.85]
    PCT_START: 0.4
    DIV_FACTOR: 10
    DECAY_STEP_LIST: [35, 45]
    LR_DECAY: 0.1
    LR_CLIP: 0.0000001

    LR_WARMUP: False
    WARMUP_EPOCH: 1

    GRAD_NORM_CLIP: 10

In [ ]:
# index files + object database (used to paste rare objects into training scenes)
%cd /content/OpenPCDet
import yaml
from easydict import EasyDict
import pcdet.datasets.custom.custom_dataset as custom_dataset
from pcdet.datasets.custom.custom_dataset import create_custom_infos
custom_dataset.Path = Path  # in case the file was imported before the patch above
cfg = EasyDict(yaml.safe_load(open('tools/cfgs/dataset_configs/vn_dataset.yaml')))
root = Path('data/custom')
create_custom_infos(dataset_cfg=cfg, class_names=CLASSES, data_path=root, save_path=root)
!ls -la data/custom

## 3. Model khởi đầu: CenterPoint-PointPillar đã huấn luyện trên toàn bộ nuScenes (OpenPCDet model zoo)

In [ ]:
# Download the nuScenes weights and map its 10-class heads onto our VN heads, class by class.
# Without them (download blocked), training still runs from scratch.
import glob, subprocess, torch
PRETRAINED_ID = '1UvGm6mROMyJzeSRu7OD1leU_YWoAZG7v'  # cbgs_dyn_pp_centerpoint (nuScenes mAP 50.0)
SRC_HEADS = [['car'], ['truck', 'construction_vehicle'], ['bus', 'trailer'], ['barrier'],
             ['motorcycle', 'bicycle'], ['pedestrian', 'traffic_cone']]
OUR_HEADS = [['Car'], ['LargeVehicle'], ['TwoWheeler'], ['Pedestrian']]  # = vn_centerpoint_pillar.yaml
SOURCE = {'Car': 'car', 'LargeVehicle': 'truck', 'TwoWheeler': 'motorcycle', 'Pedestrian': 'pedestrian'}

src = '/content/nuscenes_pp_centerpoint.pth'
if not os.path.exists(src):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
    subprocess.run(['gdown', PRETRAINED_ID, '-O', src])

if not os.path.exists(src):  # fallback: the .pth uploaded to the Drive folder
    found = glob.glob(f'{DRIVE_DIR}/*centerpoint*.pth')
    src = found[0] if found else None

PRETRAINED = None
if src:
    state = torch.load(src, map_location='cpu', weights_only=False)['model_state']
    for h, names in enumerate(SRC_HEADS):  # the assumed head layout must match the file
        assert state[f'dense_head.heads_list.{h}.hm.1.weight'].shape[0] == len(names), 'head layout differs'
    where = {c: (h, i) for h, names in enumerate(SRC_HEADS) for i, c in enumerate(names)}
    new = {k: v for k, v in state.items() if not k.startswith('dense_head.heads_list.')}
    for h, names in enumerate(OUR_HEADS):
        prefix = f'dense_head.heads_list.{where[SOURCE[names[0]]][0]}.'
        for key, value in state.items():
            if key.startswith(prefix) and '.vel.' not in key:
                new[f'dense_head.heads_list.{h}.' + key[len(prefix):]] = value.clone()
        for part in ('weight', 'bias'):
            new[f'dense_head.heads_list.{h}.hm.1.{part}'] = torch.stack([
                state[f'dense_head.heads_list.{where[SOURCE[n]][0]}.hm.1.{part}'][where[SOURCE[n]][1]] for n in names
            ])
    PRETRAINED = '/content/pretrained_vn.pth'
    torch.save({'model_state': new}, PRETRAINED)
    print('pretrained:', PRETRAINED, len(new), 'tensors')
else:
    print('Không tải được model nuScenes: huấn luyện từ đầu')

# a model of an earlier round (same classes and heads) added as input: learn on from it instead
INIT = sorted(glob.glob(f'{DRIVE_DIR}/init/checkpoint_epoch_*.pth', recursive=True), key=lambda f: int(Path(f).stem.rsplit('_', 1)[1]))
if INIT:
    state = torch.load(INIT[-1], map_location='cpu', weights_only=False)['model_state']
    heads = [state.get(f'dense_head.heads_list.{h}.hm.1.weight') for h in range(len(OUR_HEADS))]
    if all(w is not None and w.shape[0] == len(names) for w, names in zip(heads, OUR_HEADS)):
        PRETRAINED = INIT[-1]
        print('học tiếp từ model vòng trước:', PRETRAINED)
    else:
        print('bỏ qua', INIT[-1], ': khác cấu trúc head')

## 4. Huấn luyện và đánh giá

In [ ]:
# full log in /content/train.log; the cell fails if training fails
%cd /content/OpenPCDet/tools
import shutil, subprocess, time

# 1) the output dir lives on Google Drive: after a disconnect, Run all again and train.py resumes from the
#    newest checkpoint it finds there
out_root = Path('/content/OpenPCDet/output')
drive_out = Path(DRIVE_DIR) / 'output'
run_dir = drive_out / 'custom_models' / 'vn_centerpoint_pillar'
if START_FRESH:
    shutil.rmtree(run_dir, ignore_errors=True)
drive_out.mkdir(parents=True, exist_ok=True)
if not out_root.is_symlink():
    shutil.rmtree(out_root, ignore_errors=True)
    out_root.symlink_to(drive_out)
saved = sorted(run_dir.glob('*/ckpt/*.pth'), key=lambda f: f.stat().st_mtime)
print('Học tiếp từ', saved[-1].name if saved else 'đầu (chưa có checkpoint trên Drive)')

# 2) preflight (~1-2 min): build data + model, load the pretrained weights, one train step, one inference
#    batch on GPU, so a problem shows now instead of after an hour of training
PREFLIGHT = r'''
import os, sys
os.chdir('/content/OpenPCDet/tools')
sys.path.insert(0, '/content/OpenPCDet')
import torch
from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.datasets import build_dataloader
from pcdet.models import build_network, load_data_to_gpu
from pcdet.utils import common_utils
cfg_from_yaml_file('cfgs/custom_models/vn_centerpoint_pillar.yaml', cfg)
logger = common_utils.create_logger()
train_set, train_loader, _ = build_dataloader(dataset_cfg=cfg.DATA_CONFIG, class_names=cfg.CLASS_NAMES,
                                              batch_size=2, dist=False, workers=0, logger=logger, training=True)
model = build_network(model_cfg=cfg.MODEL, num_class=len(cfg.CLASS_NAMES), dataset=train_set).cuda()
if len(sys.argv) > 1:
    model.load_params_from_file(filename=sys.argv[1], logger=logger, to_cpu=False)
model.train()
batch = next(iter(train_loader))
load_data_to_gpu(batch)
ret, _, _ = model(batch)
ret['loss'].backward()
print('PREFLIGHT train step OK, loss', float(ret['loss']))
sys.path.insert(0, '/content/OpenPCDet/tools')
from train_utils.optimization import build_optimizer
from train_utils.train_utils import checkpoint_state, save_checkpoint
optimizer = build_optimizer(model, cfg.OPTIMIZATION)
optimizer.step()
save_checkpoint(checkpoint_state(model, optimizer, 1, 1), filename='/content/preflight_ckpt')
model.load_params_from_file(filename='/content/preflight_ckpt.pth', logger=logger, to_cpu=False)
print('PREFLIGHT checkpoint save + load OK')
test_set, test_loader, _ = build_dataloader(dataset_cfg=cfg.DATA_CONFIG, class_names=cfg.CLASS_NAMES,
                                            batch_size=2, dist=False, workers=0, logger=logger, training=False)
model.eval()
batch = next(iter(test_loader))
load_data_to_gpu(batch)
with torch.no_grad():
    pred_dicts, _ = model(batch)
annos = test_set.generate_prediction_dicts(batch, pred_dicts, cfg.CLASS_NAMES)
print('PREFLIGHT inference OK,', len(annos[0]['name']), 'boxes on the first frame')
'''
Path('/content/preflight.py').write_text(PREFLIGHT)
# PyTorch >= 2.6 loads with weights_only=True, which rejects the numpy scalars OpenPCDet stores in its own
# checkpoints; these files are produced by this notebook, so a full load is safe
RUN_ENV = {**os.environ, 'TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD': '1'}
check = subprocess.run([sys.executable, '/content/preflight.py'] + ([PRETRAINED] if PRETRAINED else []),
                       capture_output=True, text=True, env=RUN_ENV)
print('\n'.join(l for l in (check.stdout + check.stderr).splitlines()
                if 'PREFLIGHT' in l or 'loaded' in l or 'Error' in l or 'error' in l)[-3000:])
if check.returncode != 0:
    print((check.stdout + check.stderr)[-4000:])
    raise SystemExit('Preflight lỗi: gửi phần log ở trên (chưa tốn thời gian huấn luyện)')

# 3) the real training run, on every GPU (T4 x2: about twice as fast); a progress line every 5 minutes
def train_cmd(gpus):
    args = ['train.py', '--cfg_file', 'cfgs/custom_models/vn_centerpoint_pillar.yaml',
            '--batch_size', str(BATCH_SIZE * gpus), '--epochs', str(EPOCHS), '--workers', '2', '--extra_tag', 'colab_v5', '--max_ckpt_save_num', '3']
    if PRETRAINED:
        args += ['--pretrained_model', PRETRAINED]
    if gpus > 1:
        return [sys.executable, '-m', 'torch.distributed.run', '--nproc_per_node', str(gpus), '--master_port', '29511'] + args + ['--launcher', 'pytorch']
    return [sys.executable] + args

def run_training(gpus):
    with open('/content/train.log', 'w') as log:
        proc = subprocess.Popen(train_cmd(gpus), stdout=log, stderr=subprocess.STDOUT, env=RUN_ENV)
        while proc.poll() is None:
            time.sleep(300)
            tail = open('/content/train.log', 'rb').read()[-4000:].decode(errors='ignore')
            lines = [l.strip() for l in tail.replace('\r', '\n').splitlines() if l.strip()]
            print(time.strftime('%H:%M'), lines[-1][:200] if lines else '...', flush=True)
    return proc

GPUS = torch.cuda.device_count()
print('GPU:', GPUS)
proc = run_training(GPUS)
run_out = Path('/content/OpenPCDet/output/custom_models/vn_centerpoint_pillar')
if proc.returncode != 0 and GPUS > 1 and not list(run_out.glob('*/ckpt/checkpoint_epoch_*.pth')):
    # multi-GPU failed before the first epoch was saved: same run on one GPU
    print(open('/content/train.log', errors='ignore').read()[-3000:])
    print('Chạy nhiều GPU lỗi trước epoch đầu: chạy lại trên 1 GPU')
    shutil.copy('/content/train.log', '/content/train_multi_gpu_failed.log')
    shutil.rmtree(run_out, ignore_errors=True)
    proc = run_training(1)
!grep -E "==> Done \(loaded|Not updated|==> Loading parameters from checkpoint" /content/train.log | head -n 20
!tail -n 60 /content/train.log
assert proc.returncode == 0, 'Huấn luyện lỗi: xem log ở trên (đầy đủ trong /content/train.log)'

In [ ]:
# keep the useful outputs (checkpoints are large: only the last two)
import re, shutil
out = Path('/content/OpenPCDet/output/custom_models/vn_centerpoint_pillar/colab_v5')
RESULTS.mkdir(parents=True, exist_ok=True)
# checkpoint_epoch_<n>.pth by epoch number (the ckpt dir also holds latest_model.pth, without a number)
ckpts = sorted(out.glob('ckpt/checkpoint_epoch_*.pth'), key=lambda f: int(f.stem.rsplit('_', 1)[1]))
assert ckpts, 'Không có checkpoint: bước huấn luyện chưa chạy xong'
for ckpt in ckpts[-2:]:
    shutil.copy(ckpt, RESULTS)
for log in out.rglob('*.txt'):
    shutil.copy(log, RESULTS / log.name)
for f in out.rglob('result.pkl'):
    epoch = next((part for part in f.parts if part.startswith('epoch_')), 'eval')
    shutil.copy(f, RESULTS / f'{epoch}_result.pkl')
shutil.copy('/content/train.log', RESULTS)
print('Kết quả ở', RESULTS)
for f in sorted(RESULTS.iterdir()):
    print(f'  {f.name}  {f.stat().st_size / 2**20:.1f} MB')
!grep -hE "AP|recall" "{RESULTS}"/*.txt "{RESULTS}"/train.log | tail -n 40